In [1]:
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━ 1.0/1.3 MB 28.7 MB/s eta 0:00:01Requirement already satisfied: charset_normalizer<4,>=2 in /usr/local/lib/python3.12/dist-packages (from requests->torch-geometric) (3.4.7)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.6 MB/s eta 0:00:00


In [4]:
# -*- coding: utf-8 -*-
"""
Full PPI + Protein Complex Prediction Pipeline
=============================================
Upload these files to Colab before running:
  - proteinGraphsIndexed.pkl
  - protein_index_map.json
  - positiveEdges_indexed.csv       (your positive_edge_1024.csv, already indexed)
  - combined_stringent.txt          (Negatome: https://mips.helmholtz-muenchen.de/proj/ppi/negatome/combined_stringent.txt)
  - DT1_Dict_hippie.json
"""

# ================= INSTALL =================
# Run this cell first in Colab:
# !pip install torch-geometric

# ================= IMPORTS =================

import torch
import torch.nn as nn
import torch.nn.functional as F
import pickle
import json
import random
import numpy as np

from torch_geometric.nn import GATConv, GCNConv, global_mean_pool, SAGPooling
from torch.utils.data import DataLoader as TorchDataLoader
from sklearn.metrics import roc_auc_score, average_precision_score

In [3]:
# =====================================================
# CONFIG — FULL DATASET
# =====================================================

SUBSET     = 7499       # full dataset
HIDDEN_DIM = 128
EMBED_DIM  = 64
HEADS      = 1
EPOCHS     = 10
LR         = 1e-3
BATCH_SIZE = 256
NEG_RATIO  = 1
SEED       = 42

device = torch.device("cpu")
torch.set_num_threads(4)
print(f"Using device: cpu | Threads: {torch.get_num_threads()}")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

Using device: cpu | Threads: 4


In [5]:
# =====================================================
# STEP 2 — LOAD DATA
# =====================================================

with open("/content/proteinGraphsIndexed.pkl", "rb") as f:
    protein_graphs = pickle.load(f)
print(f"Loaded {len(protein_graphs)} protein graphs")

with open("/content/protein_index_map.json") as f:
    protein_index_map = json.load(f)
uniprot2idx = protein_index_map
print(f"Proteins in index map: {len(uniprot2idx)}")

Loaded 7499 protein graphs
Proteins in index map: 7499


In [6]:
# =====================================================
# STEP 3 — POSITIVE EDGES
# =====================================================

positive_edges = []
with open("/content/positiveEdges_indexed.csv") as f:
    next(f)
    for line in f:
        a, b = line.strip().split(",")
        a, b = int(a), int(b)
        if a < len(protein_graphs) and b < len(protein_graphs):
            positive_edges.append((a, b))
print(f"Positive edges: {len(positive_edges)}")

Positive edges: 18764


In [7]:
# =====================================================
# STEP 4 — NEGATIVE EDGES (Negatome + random top-up)
# =====================================================

N = len(protein_graphs)
negatome_edges = []
try:
    with open("/content/combined_stringent.txt") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) < 2:
                continue
            a_uni = parts[0].split("-")[0]
            b_uni = parts[1].split("-")[0]
            if a_uni in uniprot2idx and b_uni in uniprot2idx:
                i, j = uniprot2idx[a_uni], uniprot2idx[b_uni]
                if i != j and i < N and j < N:
                    negatome_edges.append((i, j))
    print(f"Negatome negatives matched: {len(negatome_edges)}")
except FileNotFoundError:
    print("WARNING: combined_stringent.txt not found — random negatives only.")

positive_set = set(positive_edges) | {(b, a) for a, b in positive_edges}
target_neg   = NEG_RATIO * len(positive_edges)
random_negs  = set((i, j) for i, j in negatome_edges)

attempts = 0
max_attempts = target_neg * 20
while len(random_negs) < target_neg and attempts < max_attempts:
    i = random.randint(0, N - 1)
    j = random.randint(0, N - 1)
    if i != j and (i, j) not in positive_set and (i, j) not in random_negs:
        random_negs.add((i, j))
    attempts += 1

negative_edges = list(random_negs)[:target_neg]
print(f"Total negative edges: {len(negative_edges)}")

Negatome negatives matched: 673
Total negative edges: 18764


In [8]:
# =====================================================
# STEP 5 — PPI EDGE INDEX (for GCN topology)
# =====================================================

all_ppi = []
for a, b in positive_edges:
    all_ppi.append([a, b])
    all_ppi.append([b, a])

ppi_edge_index = torch.tensor(all_ppi, dtype=torch.long).t().contiguous().to(device)
print(f"PPI edge_index shape: {ppi_edge_index.shape}")

PPI edge_index shape: torch.Size([2, 37528])


In [9]:
# =====================================================
# STEP 6 — TRAIN / VAL / TEST SPLITS
# =====================================================

def split_edges(edges, train=0.70, val=0.15):
    edges = list(edges)          # always copy — never mutate original
    random.shuffle(edges)
    n = len(edges)
    t = int(n * train)
    v = int(n * (train + val))
    return edges[:t], edges[t:v], edges[v:]

def make_batches(pos, neg, batch_size):
    pairs = [(a, b, 1) for a, b in pos] + [(a, b, 0) for a, b in neg]
    random.shuffle(pairs)
    for i in range(0, len(pairs), batch_size):
        chunk = pairs[i:i+batch_size]
        src   = torch.tensor([p[0] for p in chunk], dtype=torch.long)
        dst   = torch.tensor([p[1] for p in chunk], dtype=torch.long)
        lbl   = torch.tensor([p[2] for p in chunk], dtype=torch.float)
        yield src, dst, lbl

pos_train, pos_val, pos_test = split_edges(positive_edges)
neg_train, neg_val, neg_test = split_edges(negative_edges)

print(f"Train  — pos: {len(pos_train)}, neg: {len(neg_train)}")
print(f"Val    — pos: {len(pos_val)},   neg: {len(neg_val)}")
print(f"Test   — pos: {len(pos_test)},  neg: {len(neg_test)}")

Train  — pos: 13134, neg: 13134
Val    — pos: 2815,   neg: 2815
Test   — pos: 2815,  neg: 2815


In [10]:
# =====================================================
# MODELS  (minimal changes from your original)
# =====================================================

class GAT1(nn.Module):
    """Structure-aware encoder for a single protein graph."""

    def __init__(self, input_dim=24, hidden_dim=HIDDEN_DIM, heads=HEADS):
        super().__init__()
        self.fc1   = nn.Linear(input_dim, hidden_dim)
        self.conv1 = GATConv(hidden_dim, hidden_dim, heads=heads, edge_dim=1)
        self.conv2 = GATConv(hidden_dim, hidden_dim, heads=heads, edge_dim=1)
        self.conv3 = GATConv(hidden_dim, hidden_dim, heads=heads, edge_dim=1)
        self.pool1 = SAGPooling(hidden_dim)
        self.pool2 = SAGPooling(hidden_dim)
        self.pool3 = SAGPooling(hidden_dim)
        self.bn1   = nn.BatchNorm1d(hidden_dim)
        self.bn2   = nn.BatchNorm1d(hidden_dim)
        self.bn3   = nn.BatchNorm1d(hidden_dim)

    def forward(self, data):
        x          = data.x
        edge_index = data.edge_index
        edge_attr  = data.edge_attr

        # dummy batch: all nodes belong to one graph
        batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)

        x = self.fc1(x)

        # Layer 1
        x_res = x
        x     = self.conv1(x, edge_index, edge_attr=edge_attr)
        x     = self.bn1(x)
        x     = F.relu(x + x_res)
        x, edge_index, edge_attr, batch, _, _ = self.pool1(
            x, edge_index, edge_attr=edge_attr, batch=batch)

        # Layer 2
        x_res = x
        x     = self.conv2(x, edge_index, edge_attr=edge_attr)
        x     = self.bn2(x)
        x     = F.relu(x + x_res)
        x, edge_index, edge_attr, batch, _, _ = self.pool2(
            x, edge_index, edge_attr=edge_attr, batch=batch)

        # Layer 3
        x_res = x
        x     = self.conv3(x, edge_index, edge_attr=edge_attr)
        x     = self.bn3(x)
        x     = F.relu(x + x_res)
        x, edge_index, edge_attr, batch, _, _ = self.pool3(
            x, edge_index, edge_attr=edge_attr, batch=batch)

        return global_mean_pool(x, batch)   # [1, hidden_dim]

In [11]:
class JointGAT_GCN(nn.Module):
    """
    Full model:
      GAT1 per protein  →  project 512→64  →  GCN over PPI graph
    The only change from your original: hidden_dim is now a parameter
    so we can reduce it to 128 for Colab.
    """

    def __init__(self, hidden_dim=HIDDEN_DIM, embed_dim=EMBED_DIM):
        super().__init__()
        self.gat     = GAT1(hidden_dim=hidden_dim)
        self.project = nn.Linear(hidden_dim, embed_dim)
        self.gcn1    = GCNConv(embed_dim, embed_dim)
        self.gcn2    = GCNConv(embed_dim, embed_dim)

    def encode_all(self, protein_graphs, ppi_edge_index):
        """
        Encode all proteins once → matrix X [N, embed_dim]
        Then refine with GCN over PPI topology.
        """
        embeddings = []
        for g in protein_graphs:
            g   = g.to(device)
            emb = self.gat(g)           # [1, hidden_dim]
            emb = self.project(emb)     # [1, embed_dim]
            embeddings.append(emb)

        X = torch.cat(embeddings, dim=0)             # [N, embed_dim]
        X = F.relu(self.gcn1(X, ppi_edge_index))
        X = self.gcn2(X, ppi_edge_index)            # [N, embed_dim]
        return X

    def forward(self, protein_graphs, ppi_edge_index):
        return self.encode_all(protein_graphs, ppi_edge_index)

In [12]:
# =====================================================
# STEP 8 — TRAINING
# =====================================================

model     = JointGAT_GCN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

def compute_loss_and_scores(z, src, dst, lbl):
    scores = (z[src] * z[dst]).sum(dim=1)
    probs  = torch.sigmoid(scores)
    loss   = F.binary_cross_entropy(probs, lbl.to(device))
    return loss, probs.detach().cpu(), lbl.cpu()

def evaluate(z, pos_edges, neg_edges):
    pairs = [(a, b, 1) for a, b in pos_edges] + [(a, b, 0) for a, b in neg_edges]
    if not pairs:
        return 0.0, 0.0
    src = torch.tensor([p[0] for p in pairs], dtype=torch.long)
    dst = torch.tensor([p[1] for p in pairs], dtype=torch.long)
    lbl = [p[2] for p in pairs]
    with torch.no_grad():
        scores = (z[src] * z[dst]).sum(dim=1)
        probs  = torch.sigmoid(scores).numpy()
    return roc_auc_score(lbl, probs), average_precision_score(lbl, probs)

def get_gat_embeddings(model, protein_graphs):
    model.eval()
    embeddings = []
    with torch.no_grad():
        for i, g in enumerate(protein_graphs):
            g   = g.to(device)
            emb = model.gat(g)
            emb = model.project(emb)
            embeddings.append(emb.cpu())
            if (i + 1) % 500 == 0:
                print(f"  GAT encoded {i+1}/{len(protein_graphs)}")
    return torch.cat(embeddings, dim=0)

def make_batches(pos, neg, batch_size):
    pairs = [(a, b, 1) for a, b in pos] + [(a, b, 0) for a, b in neg]
    random.shuffle(pairs)
    for i in range(0, len(pairs), batch_size):
        chunk = pairs[i:i+batch_size]
        src   = torch.tensor([p[0] for p in chunk], dtype=torch.long)
        dst   = torch.tensor([p[1] for p in chunk], dtype=torch.long)
        lbl   = torch.tensor([p[2] for p in chunk], dtype=torch.float)
        yield src, dst, lbl

print("\n========== TRAINING ==========")
for epoch in range(1, EPOCHS + 1):
    print(f"\n--- Epoch {epoch} ---")

    # GAT: encode all proteins once, detached
    print("  Encoding proteins with GAT...")
    gat_embs     = get_gat_embeddings(model, protein_graphs)
    gat_embs_dev = gat_embs.to(device)

    # ============================================================
    # GAT FINE-TUNING STEP (insert here)
    # ============================================================
    model.train()
    GAT_SAMPLE = 100
    sample_idx = random.sample(range(len(protein_graphs)), GAT_SAMPLE)
    sample_set = set(sample_idx)
    sample_pos = [(a, b) for a, b in pos_train if a in sample_set and b in sample_set][:50]
    sample_neg = [(a, b) for a, b in neg_train if a in sample_set and b in sample_set][:50]

    if len(sample_pos) > 1 and len(sample_neg) > 1:
        local_embs = {}
        for idx in sample_idx:
            g   = protein_graphs[idx].to(device)
            emb = model.gat(g)
            emb = model.project(emb)
            local_embs[idx] = emb.squeeze(0)

        pairs = [(a, b, 1) for a, b in sample_pos] + [(a, b, 0) for a, b in sample_neg]
        gat_loss = 0
        for a, b, lbl in pairs:
            if a in local_embs and b in local_embs:
                score = (local_embs[a] * local_embs[b]).sum()
                prob  = torch.sigmoid(score)
                gat_loss += F.binary_cross_entropy(prob.unsqueeze(0),
                                                   torch.tensor([float(lbl)]))
        if gat_loss > 0:
            optimizer.zero_grad()
            gat_loss.backward()
            optimizer.step()
            print(f"  GAT fine-tune loss: {gat_loss.item():.4f}")
    # ============================================================
    # END GAT FINE-TUNING
    # ============================================================

    # GCN: train with gradients
    model.train()
    total_loss, steps = 0, 0

    for src, dst, lbl in make_batches(pos_train, neg_train, BATCH_SIZE):
        optimizer.zero_grad()
        X    = F.relu(model.gcn1(gat_embs_dev, ppi_edge_index))
        z    = model.gcn2(X, ppi_edge_index)
        loss, _, _ = compute_loss_and_scores(z, src, dst, lbl)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        steps      += 1

    # Validation
    model.eval()
    with torch.no_grad():
        gat_embs_val = get_gat_embeddings(model, protein_graphs)
        X_val = F.relu(model.gcn1(gat_embs_val, ppi_edge_index))
        z_val = model.gcn2(X_val, ppi_edge_index)
    val_auc, val_ap = evaluate(z_val, pos_val, neg_val)

    print(f"Epoch {epoch:02d} | Loss: {total_loss/steps:.4f} | "
          f"Val AUC: {val_auc:.4f} | Val AP: {val_ap:.4f}")

    # Checkpoint every 2 epochs
    if epoch % 2 == 0:
        torch.save(model.state_dict(), f"/content/checkpoint_epoch{epoch}.pt")
        print(f"  → Checkpoint saved")


========== TRAINING ==========

--- Epoch 1 ---
  Encoding proteins with GAT...
  GAT encoded 500/7499
  GAT encoded 1000/7499
  GAT encoded 1500/7499
  GAT encoded 2000/7499
  GAT encoded 2500/7499
  GAT encoded 3000/7499
  GAT encoded 3500/7499
  GAT encoded 4000/7499
  GAT encoded 4500/7499
  GAT encoded 5000/7499
  GAT encoded 5500/7499
  GAT encoded 6000/7499
  GAT encoded 6500/7499
  GAT encoded 7000/7499
  GAT fine-tune loss: 4.7299
  GAT encoded 500/7499
  GAT encoded 1000/7499
  GAT encoded 1500/7499
  GAT encoded 2000/7499
  GAT encoded 2500/7499
  GAT encoded 3000/7499
  GAT encoded 3500/7499
  GAT encoded 4000/7499
  GAT encoded 4500/7499
  GAT encoded 5000/7499
  GAT encoded 5500/7499
  GAT encoded 6000/7499
  GAT encoded 6500/7499
  GAT encoded 7000/7499
Epoch 01 | Loss: 49.9951 | Val AUC: 0.8927 | Val AP: 0.8762

--- Epoch 2 ---
  Encoding proteins with GAT...
  GAT encoded 500/7499
  GAT encoded 1000/7499
  GAT encoded 1500/7499
  GAT encoded 2000/7499
  GAT encoded 25

In [13]:
# =====================================================
# STEP 9 — TEST
# =====================================================

model.eval()
with torch.no_grad():
    gat_embs_test = get_gat_embeddings(model, protein_graphs)
    X_test  = F.relu(model.gcn1(gat_embs_test, ppi_edge_index))
    z_final = model.gcn2(X_test, ppi_edge_index)

test_auc, test_ap = evaluate(z_final, pos_test, neg_test)
print(f"\n===== TEST RESULTS =====")
print(f"Test AUC: {test_auc:.4f}")
print(f"Test AP : {test_ap:.4f}")

# Save
torch.save(model.state_dict(), "/content/joint_model.pt")
torch.save(z_final, "/content/protein_embeddings.pt")
print(f"\nSaved model and embeddings. z shape: {z_final.shape}")

  GAT encoded 500/7499
  GAT encoded 1000/7499
  GAT encoded 1500/7499
  GAT encoded 2000/7499
  GAT encoded 2500/7499
  GAT encoded 3000/7499
  GAT encoded 3500/7499
  GAT encoded 4000/7499
  GAT encoded 4500/7499
  GAT encoded 5000/7499
  GAT encoded 5500/7499
  GAT encoded 6000/7499
  GAT encoded 6500/7499
  GAT encoded 7000/7499

===== TEST RESULTS =====
Test AUC: 0.3336
Test AP : 0.4627

Saved model and embeddings. z shape: torch.Size([7499, 64])
